In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer # for text data tokenization
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import linear_kernel # for similarity calculation
from sklearn.decomposition import TruncatedSVD
from difflib import get_close_matches
plt.style.use('ggplot') # estilo dos gráficos
pd.set_option('display.max_columns', 200) # quantidade de colunas que aparecem do nosso dataset
import cudf



#dataset_movies = pd.read_csv(r'C:\Users\BasilioCosta\Desktop\ines_ambiente_trabalho\mestrado\projeto_integrado\repositorio\Projeto-RecSys-1\exploracao_ines\movies.csv', sep = ';', encoding = 'ISO-8859-1')
# coloquei encoding='ISO-8859-1' porque existem caracteres especiais no dataset que impedem que ele corra em UTF-8

#dataset_ratings = pd.read_csv(r'C:\Users\BasilioCosta\Desktop\ines_ambiente_trabalho\mestrado\projeto_integrado\repositorio\Projeto-RecSys-1\exploracao_ines\ratings.csv', sep = ';')
# este já não precisou de um encoding diferente, porque é um dataset numérico

#datajoin = pd.concat([dataset_movies, dataset_ratings], axis = 1)

""" for col in datajoin.columns: # aqui confirmamos que a juncao dos datasets funcionou bem
    print(list(col))  """

# shift + alt + A faz o tipo de comentário acima

dataset_movies = pd.read_csv(r'/content/drive/MyDrive/datasets/projeto/movies.csv', sep = ';', encoding = 'ISO-8859-1')

dataset_ratings = pd.read_csv(r'/content/drive/MyDrive/datasets/projeto/ratings.csv', sep = ';')

datajoin = pd.concat([dataset_movies, dataset_ratings], axis = 1)

# ANÁLISE DO DATASET

#print(datajoin.shape) # temos inicialmente 85855 linhas e 71 colunas
#print(datajoin.head(20)) # 20 primeiras linhas do dataset
# como o pd por defeito não nos deixa ver as colunas todas e estamos a obter "imdb_title_id  ... non_us_voters_votes",
# então ativamos a condicao das linhas iniciais: pd.set_option('display.max_columns', 200)
#print(datajoin.columns) # nome das colunas
#print(datajoin.dtypes) # tipo de variáveis de cada coluna

#print(datajoin.describe) # similar aos outros




# RETIRAR COLUNAS DESNECESSÁRIAS PARA O CASO DE ESTUDO
df = datajoin[[#'imdb_title_id',
        'title', 'original_title',
        'year', 'date_published', 'genre', 'duration',
        'country', 'language', 'director', 'writer', 'production_company', 'actors',
        'description',
        'avg_vote', 'votes',
        #'budget', 'usa_gross_income', 'worlwide_gross_income', 'metascore',
        #'reviews_from_users', 'reviews_from_critics',
        #'imdb_title_id',
        'weighted_average_vote', 'total_votes', 'mean_vote', 'median_vote',
        'votes_10', 'votes_9', 'votes_8', 'votes_7', 'votes_6', 'votes_5',
        'votes_4', 'votes_3', 'votes_2', 'votes_1',
        #'allgenders_0age_avg_vote', 'allgenders_0age_votes',
        'allgenders_18age_avg_vote',
        'allgenders_18age_votes', 'allgenders_30age_avg_vote',
        'allgenders_30age_votes', 'allgenders_45age_avg_vote',
        'allgenders_45age_votes', 'males_allages_avg_vote',
        'males_allages_votes',
        #'males_0age_avg_vote', 'males_0age_votes',
        'males_18age_avg_vote', 'males_18age_votes', 'males_30age_avg_vote',
        'males_30age_votes', 'males_45age_avg_vote', 'males_45age_votes',
        'females_allages_avg_vote', 'females_allages_votes',
        #'females_0age_avg_vote', 'females_0age_votes',
        'females_18age_avg_vote', 'females_18age_votes', 'females_30age_avg_vote', 'females_30age_votes',
        'females_45age_avg_vote', 'females_45age_votes',
        'top1000_voters_rating', 'top1000_voters_votes', 'us_voters_rating',
        'us_voters_votes', 'non_us_voters_rating', 'non_us_voters_votes']]

# retirei as colunas que não faziam sentido para o estudo que vamos fazer ou repetidas
# (provavelmente ainda vale a pena tirar mais) - mantive assim para ter nocao do que foi
# retirado




# RENOMEAR AS COLUNAS
df = df.rename(columns = {'allgenders_18age_avg_vote' : 'allgenders_18_to_30_avg_vote',
        'allgenders_18age_votes' : 'allgenders_18_to_30_votes',
        'allgenders_30age_avg_vote' : 'allgenders_30_to_45_avg_vote',
        'allgenders_30age_votes' : 'allgenders_30_to_45_votes',
        'allgenders_45age_avg_vote' : 'allgenders_45_and_above_avg_vote',
        'allgenders_45age_votes' : 'allgenders_45_and_above_votes',
        'males_18age_avg_vote' : 'males_18_to_30_avg_vote',
        'males_18age_votes' : 'males_18_to_30_votes',
        'males_30age_avg_vote' : 'males_30_to_45_avg_vote',
        'males_30age_votes' : 'males_30_to_45_votes',
        'males_45age_avg_vote' : 'males_45_and_above_avg_vote',
        'males_45age_votes' : 'males_45_and_above_votes',
        'females_18age_avg_vote' : 'females_18_to_30_avg_vote',
        'females_18age_votes' : 'females_18_to_30_votes',
        'females_30age_avg_vote' : 'females_30_to_45_avg_vote',
        'females_30age_votes' : 'females_30_to_45_votes',
        'females_45age_avg_vote' : 'females_45_and_above_avg_vote',
        'females_45age_votes' : 'females_45_and_above_votes'})




# MISSING VALUES:

#print(df.isna().sum()) # aqui sabemos quantos missing values temos em cada coluna




# MISSING VALUES - COLUNA YEAR:

#print(df['year'].isnull().sum()) # existe 1 missing value na coluna 'year'
ind_missval_year = df[df['year'].isnull()].index[0] # a linha onde está o missing value é a 83917

# o missing value de 'year' pode ser substituído pelo valor da coluna 'date_published:
valor_date_published = df.at[ind_missval_year, 'date_published']  # Obtém o valor da coluna 'date_published'
                                                                  # neste índice

# Usamos uma expressão regular para extrair o ano, porque o resultado era TV Movie 2019 e só queremos o 2019
ano = re.search(r'\b\d{4}\b', valor_date_published)

# Se o ano for encontrado, substituir o valor na coluna 'year'
if ano:
    df.at[ind_missval_year, 'year'] = int(ano.group())  # Substitui o valor de 'year' com o ano extraído, ou seja, não
                                                        # existem mais missing values na coluna 'year'

df = df.astype({'year' : 'int'}) # evita que os números apareçam como floats




# MISSING VALUES - COLUNAS A PREENCHAR:

coluna_inicio = 'avg_vote'  # A partir de onde é para preencher os missing values baseado numa regra

colunas_a_preencher = df.loc[:, coluna_inicio:].columns # pegar em todas as colunas a partir da 'coluna_inicio'
                                                        # para saber o que temos de preencher

df[colunas_a_preencher] = df[colunas_a_preencher].fillna(df[colunas_a_preencher].mean())


# as colunas 'country', 'language', 'director', 'writer', 'production_company', 'actors' ficaram por tratar no que toca
# a missing values. Penso que este tratamento pode ser feito com base noutras colunas, mas não o fiz, porque honestamente
# não saberia por onde começar. Talvez o que faça mais sentido é cruzar o país com a língua e caso ambos possuam missing
# values na mesma linha, então poderíamos pensar quem foi o diretor e tentar descobrir o pais e a língua, mas isso é
# subjetivo, porque pode não ter nada a ver. E ainda por cima não podemos inferir o nome do diretor com base no país
# por haver países que têm nomes parecidos. Sei lá :)




# TRATAMENTO DE DADOS - COLUNA DATE_PUBLISHED:

# Nesta parte decidi trabalhar nas datas da coluna date_published. Tinha algumas datas que estavam invertidas, por exemplo
# 1894-10-09 e outras que só tinham o ano, por exemplo, 2019. Assim, inverti as que estavam invertidas inicialmente e as que
# só tinham o ano passaram a ser 01-01-2019

# Função para verificar se a string está no formato dd-mm-yyyy
def is_valid_date_format(date_str):
    return bool(re.match(r'^\d{2}-\d{2}-\d{4}$', date_str))

# Função para inverter a data conforme as regras
def transform_date_format(x):

    # Caso o valor já esteja no formato dd-mm-yyyy
    if isinstance(x, str) and is_valid_date_format(x):
        return x  # Não faz nada e mantém o valor como está

    # Caso o valor esteja no formato yyyy-mm-dd
    elif isinstance(x, str) and re.match(r'^\d{4}-\d{2}-\d{2}$', x):

        # Converte para o formato dd-mm-yyyy
        date = pd.to_datetime(x)
        return date.strftime('%d-%m-%Y')  # Retorna o valor como dd-mm-yyyy

    # Caso o valor seja apenas o ano
    elif isinstance(x, str) and re.match(r'^\d{4}$', x):
        return f"01-01-{x}"  # Cria uma data com dia 01 e mês 01 - aqui pode tornar-se aleatório, se quisermos

    return x  # Retorna o valor original caso não caia em nenhum dos casos anteriores, pelo que as que já estavam bem mantém-se

# Aplica a função à coluna 'date_published'
df['date_published'] = df['date_published'].apply(transform_date_format)




# LINHAS DUPLICADAS - INEXISTENTES:

# apesar de haver muitas linhas com valores que parecem estar duplicados, na realidade, se formos a comparar os valores
# 'title', 'year' e 'language', percebemos que as linhas apenas parecem estar duplicadas; existem filmes que têm o mesmo
# nome mas que foram publicados em anos diferentes ou que estão em línguas diferentes

#print(df.loc[df.duplicated(subset = ['title', 'year', 'language'])])




# ANÁLISE ESTATÍSTICA DOS DADOS:

# PARA OS DADOS NUMÉRICOS:

coluna_inicio = 'avg_vote'  # A partir de onde é para preencher os missing values baseado numa regra

colunas_a_preencher = df.loc[:, coluna_inicio:].columns # pegar em todas as colunas a partir da 'coluna_inicio'
                                                        # para saber o que temos de preencher

medias_dict = df[colunas_a_preencher].mean().round(3).to_dict() # medias das colunas numéricas
                                                                # arredondadas para 3 casas decimais
medianas_dict = df[colunas_a_preencher].median().to_dict() # mediana das colunas numéricas

moda_dict = df[colunas_a_preencher].mode().iloc[0].to_dict() # moda das colunas numéricas
                                                             # como a moda pode ser mais do que uma
                                                             # pegamos no primeiro valor mais frequente
                                                             # usando .iloc[0]




# PARA OS DADOS DE TEXTO:

colunas_a_processar = ['genre', 'country', 'language', 'director', 'writer', 'production_company', 'actors']
# achei que estas fizessem mais sentido

# Armazenar os resultados
listas_unicas = {}  # Armazena os valores de cada coluna, apenas um de cada
mais_comuns = {}  # Armazena os 3 mais comuns

# Função para processar cada coluna
def processar_coluna(df, coluna):
    todos_os_valores = df[coluna].dropna().str.split(', ').explode() # aqui remove os missing values
    # separa os valores de cada linha por vírgulas e coloca-os em linhas diferentes

    lista_unica = sorted(todos_os_valores.unique().tolist())  # Lista ordenada dos valores que existem
    contagem = Counter(todos_os_valores)  # Freqência de cada valor
    mais_comuns = contagem.most_common(3)  # 3 mais comuns

    return lista_unica, mais_comuns

# Aplicar a função a cada coluna e armazenar os resultados
for coluna in colunas_a_processar:
    listas_unicas[coluna], mais_comuns[coluna] = processar_coluna(df, coluna)

"""
for coluna in colunas_a_processar:
    print(f"\nColuna: {coluna}")
    print(f"Valores únicos ordenados: {listas_unicas[coluna]}")
    print(f"Top 3 mais comuns: {mais_comuns[coluna]}")
"""

# para analisar cada coluna mais vale comentar as restantes no colunas_a_processar, porque o
# output é enorme; na lista dos países tem valores meio estranhos; nas colunas
# 'director', 'writer' e 'actors' podem haver valores que não estejam a falar da
# mesma pessoa porque existem pessoas com o mesmo nome



"""
# ANÁLISE GRÁFICA:

# Coluna year:

graph_year_2009 = df['year'].value_counts().sort_index(ascending = True).head(10).plot(kind = 'bar', color = 'purple')

graph_year_2009.set_title('Movies released since 2009', color = 'purple', fontweight = 'bold')
graph_year_2009.set_xlabel('Year', color = 'purple')
graph_year_2009.set_ylabel('Number of Movies Launched', color = 'purple')
graph_year_2009.set_facecolor('lavender') # altera a cor dos quadradinhos por trás do gráfico


plt.gcf().set_facecolor('lavender')  # Altera a cor do fundo do gráfico em volta dos quadradinhos

#plt.show()

graph_year_1923 = df['year'].value_counts().sort_index(ascending = True).tail(10).plot(kind = 'bar', color = 'purple')

graph_year_1923.set_title('Movies released until 1923', color = 'purple', fontweight = 'bold')
graph_year_1923.set_xlabel('Year', color = 'purple')
graph_year_1923.set_ylabel('Number of Movies Launched', color = 'purple')
graph_year_1923.set_facecolor('lavender')

plt.gcf().set_facecolor('lavender')

#plt.show()

graph_year = df['year'].value_counts().sort_index().plot(kind='line', marker='o', color = 'purple')

graph_year.set_title('Movies released', color = 'purple', fontweight = 'bold')
graph_year.set_xlabel('Year', color = 'purple')
graph_year.set_ylabel('Number of Movies Launched', color = 'purple')
graph_year.set_facecolor('lavender') # altera a cor dos quadradinhos por trás do gráfico

plt.gcf().set_facecolor('lavender')

#plt.show()




# GRÁFICO DA MÉDIA DE VOTOS POR FAIXA ETÁRIA EM CADA GÉNERO:

# Criar uma cópia do DataFrame removendo valores nulos e separando os géneros
df_exploded = df.dropna(subset=['genre']).copy()
df_exploded['genre'] = df_exploded['genre'].str.split(', ')  # Separar géneros
df_exploded = df_exploded.explode('genre')  # Explodir para várias linhas

# Somar os votos por género e faixa etária
votos_por_genero = df_exploded.groupby('genre')[[
    'allgenders_18_to_30_votes',
    'allgenders_30_to_45_votes',
    'allgenders_45_and_above_votes'
]].sum()

# Criar gráfico de barras empilhadas
votos_por_genero.plot(kind = 'bar', stacked = True, figsize = (12, 6), colormap = 'viridis')

# Personalizar gráfico
plt.title('Votes Distribution by Genre and Age Group', fontsize = 14, fontweight = 'bold', color = 'purple')
plt.xlabel('Genre', color = 'purple')
plt.ylabel('Total Votes', color = 'purple')
plt.xticks(rotation = 45)  # Rodar os nomes do eixo dos x para melhor visualização
plt.legend(['18-30 years', '30-45 years', '45+ years'], title = "Age Group")

plt.gca().set_facecolor('paleturquoise')  # Cor dos quadrados do gráfico
plt.gcf().set_facecolor('paleturquoise')

#plt.show()
"""

'\n# ANÁLISE GRÁFICA:\n\n# Coluna year:\n\ngraph_year_2009 = df[\'year\'].value_counts().sort_index(ascending = True).head(10).plot(kind = \'bar\', color = \'purple\')\n\ngraph_year_2009.set_title(\'Movies released since 2009\', color = \'purple\', fontweight = \'bold\')\ngraph_year_2009.set_xlabel(\'Year\', color = \'purple\')\ngraph_year_2009.set_ylabel(\'Number of Movies Launched\', color = \'purple\')\ngraph_year_2009.set_facecolor(\'lavender\') # altera a cor dos quadradinhos por trás do gráfico\n\n\nplt.gcf().set_facecolor(\'lavender\')  # Altera a cor do fundo do gráfico em volta dos quadradinhos\n\n#plt.show()\n\ngraph_year_1923 = df[\'year\'].value_counts().sort_index(ascending = True).tail(10).plot(kind = \'bar\', color = \'purple\')\n\ngraph_year_1923.set_title(\'Movies released until 1923\', color = \'purple\', fontweight = \'bold\')\ngraph_year_1923.set_xlabel(\'Year\', color = \'purple\')\ngraph_year_1923.set_ylabel(\'Number of Movies Launched\', color = \'purple\')\ngrap

Recommendation System based on the titles similarity

In [ ]:
### RECOMMENDATION SYSTEM WITH COSINE SIMILARITY

# tokenize text data - to work with text data, tokenization is required
# tokenazation is a way of breaking down the text into smaller units called
# tokens. In this example a token is a word. TDIDF is used to tokenized
# description column. TF-IDF (Term Frequency - Inverse Document Frequency)
# measures how relevant a word is to a document in a collection of documents.abs

# Function to check if a name exists in the dataset (case insensitive)
def name_exists(name, column, df):

    # Convert the name to lowercase for case-insensitive comparison
    all_names = df[column].dropna().apply(lambda x: [n.strip().lower() for n in x.split(', ')]) # removes the missing values
    # of the column; divides the names separated by commas and transforms everything in lowcases, removing spaces

    flat_names = set([n for sublist in all_names for n in sublist]) # creates a list without duplicates

    return name.lower() in flat_names


# Function to suggest full names based on part of the name or surname
def suggest_exact_names(name, column, df):

    # Transform all values in the column into lists of names and surnames, cleaning the spaces
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    # Flatten the list and remove duplicates
    flat_names = list(set([entry for sublist in all_names for entry in sublist]))

    # Use difflib to find similar names
    suggestions = get_close_matches(name, flat_names, n = 5, cutoff = 0.8) # only suggests names with 80% or more similarity

    # If no suggestions are found, search for the corresponding full name
    if not suggestions:
        suggestions = [entry for entry in flat_names if name.lower() in entry.lower()]

    return suggestions

# Function to get the correct name from the dataset, considering capitalization
def get_correct_name(name, column, df):
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    for sublist in all_names: # goes trought all of the names
        for real_name in sublist: # compares the given name with the dataset ones

            if real_name.lower() == name.lower(): # if it matches, returns the name thats on the dataset
                return real_name

    return None # if theres not a match then it returns none

# Function to collect user preferences
def get_user_preferences_console(df):

    # === GENRES ===
    # Show unique genres available in the dataset
    available_genres = df['genre'].str.split(', ').explode().unique()  # Extract unique genres
    print("Genres available in the dataset:")

    for idx, genre in enumerate(available_genres, 1):
        print(f"{idx}. {genre}")

    # Ask the user to choose up to 3 genres
    chosen_genres = []

    while len(chosen_genres) < 1 or len(chosen_genres) > 3:
        genre_choice = input(f"Choose one of the genres above (up to 3 genres), separating by commas: ")

        # Clean and split the input
        genre_list = [genre.strip() for genre in genre_choice.split(",")]

        # Check if all chosen genres are available in the dataset and haven't been chosen yet
        valid_genres = [g for g in genre_list if g in available_genres and g not in chosen_genres]

        # If valid genres
        if valid_genres:
            chosen_genres.extend(valid_genres)

        else:
            print("Invalid choice or genres already selected. Try again.")

        # Limit to 3 genres
        if len(chosen_genres) > 3:
            print("You chose more than 3 genres. We will limit it to 3 genres.")
            chosen_genres = chosen_genres[:3]

    # === ACTORS ===
    chosen_actors = []

    while len(chosen_actors) < 3:
        actor_input = input(f"What is you actor number {len(chosen_actors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if actor_input == "Ignore":
            break

        if name_exists(actor_input, 'actors', df):
            correct_name = get_correct_name(actor_input, 'actors', df)
            chosen_actors.append(correct_name)

        else:
            print(f"The actor '{actor_input}' wasn't found in the dataset.")
            suggestions = suggest_exact_names(actor_input, 'actors', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_actors:
        print("The user didn't choose any actor.")

    else:
        print("Chosen actors:", chosen_actors)

    # === DIRECTOR ===
    chosen_directors = []

    while len(chosen_directors) < 3:
        director_input = input(f"What is your director number {len(chosen_directors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if director_input == "Ignore":
            break

        if name_exists(director_input, 'director', df):
            correct_director = get_correct_name(director_input, 'director', df)
            chosen_directors.append(correct_director)


        else:
            print(f"The director '{director_input}' was not found in the dataset.")
            suggestions = suggest_exact_names(director_input, 'director', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_directors:
        print("The user didn't choose any director.")

    else:
        print("Chosen director:", chosen_directors)


    combined_input = ' '.join(chosen_genres + chosen_actors + chosen_directors)

    return combined_input

"""
    # === LANGUAGES ===
    # Show unique languages available in the dataset
    available_languages = df['language'].str.split(', ').explode().unique()  # Extract unique languages
    print("Languages available in the dataset:")

    for idx, language in enumerate(available_languages, 1):
        print(f"{idx}. {language}")

    # Ask the user to choose up to 3 languages
    chosen_languages = []

    while len(chosen_languages) < 1 or len(chosen_languages) > 3:
        language_choice = input("Choose one of the languages above (up to 3 languages), separating by commas: ")

        # Clean and split the input
        language_list = [lang.strip() for lang in language_choice.split(",")]

        # Check if all chosen languages are available in the dataset and haven't been chosen yet
        valid_languages = [l for l in language_list if l in available_languages and l not in chosen_languages]

        # If valid languages
        if valid_languages:
            chosen_languages.extend(valid_languages)
        else:
            print("Invalid choice or languages already selected. Try again.")

        # Limit to 3 languages
        if len(chosen_languages) > 3:
            print("You chose more than 3 languages. We will limit it to 3 languages.")
            chosen_languages = chosen_languages[:3]


    # === DESCRIPTION ===
    description = input("What type of plot do you prefer? (e.g., Sci-Fi, Romance, Adventure): ").strip()

    # Return all the collected data
    return f"{', '.join(chosen_genres)} {', '.join(chosen_actors)} {', '.join(chosen_directors)} {description}"
"""

# the upper part is not working - IMPROVE THIS SYSTEM

# Prepare data for recommendation
selected_columns = ['original_title', 'genre', 'director', 'actors', 'description']
df_selected = df[selected_columns].astype(str).head(10000)

df_selected['combined_features'] = df_selected.apply(lambda x: ' '.join(x), axis = 1)

# Text Vectorization (TF-IDF)
def create_feature_vectors(df):
    vectorizer = TfidfVectorizer(stop_words = 'english')
    feature_vectors = vectorizer.fit_transform(df['combined_features'])

    # Dimensionality reduction using TruncatedSVD (SVD) to improve performance
    svd = TruncatedSVD(n_components = 300, random_state = 42)
    reduced_features = svd.fit_transform(feature_vectors)

    return vectorizer, svd, feature_vectors, reduced_features

# Function to compute similarity
def compute_similarity(reduced_features):
    return cosine_similarity(reduced_features)

# Vectors and similarity matrix
vectorizer, svd, feature_vectors, reduced_features = create_feature_vectors(df_selected)
similarity_matrix = compute_similarity(reduced_features)



# Function to recommend movies based on the title
def recommend_movies(movie_title, num_recommendations = 5):
    if movie_title not in df_selected['original_title'].values:
        # If no movie is found, print suggestions and return an empty list
        print("Movie not found. Try another title." , end = " ")

        # Suggest similar titles if available
        suggestions = get_close_matches(movie_title, df_selected['original_title'].tolist(), n = 5, cutoff = 0.5)

        if suggestions:
            print(", ".join(suggestions))  # Print suggestions in a horizontal format
        else:
            print("No suggestions available.")  # Inform that there are no suggestions

        return []  # Return an empty list instead of None

    movie_index = df_selected[df_selected['original_title'] == movie_title].index[0]
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:num_recommendations + 1]
    recommended_movies = [df_selected.iloc[i[0]]['original_title'] for i in similarity_scores]

    return recommended_movies


# === Main function of recommended by preferences ===

def recommend_by_preferences(user_input, df_selected, vectorizer, svd, feature_vectors, num_recommendations = 5):
    user_vector = vectorizer.transform([user_input])
    user_vector_reduced = svd.transform(user_vector)
    similarity_scores = cosine_similarity(user_vector_reduced, feature_vectors).flatten()
    top_indices = similarity_scores.argsort()[::-1][:num_recommendations]
    recommended_movies = df_selected.iloc[top_indices]['original_title'].tolist()
    return recommended_movies

# === Data preparation ===
# TF-IDF + SVD
vectorizer = TfidfVectorizer(stop_words = 'english')
feature_vectors = vectorizer.fit_transform(df_selected['combined_features'])
svd = TruncatedSVD(n_components = 300, random_state = 42)
reduced_features = svd.fit_transform(feature_vectors)


# === Recommendation by title ===
while True:
    # Ask the user to provide a movie title for recommendations
    print("\n=== Recommendation System based on Title ===")
    movie_title = input("\nEnter the title of a movie for similar recommendations (or 'exit' to quit): ")

    if movie_title.lower() == 'exit':
        print("Thank you for using the recommendation system! Goodbye!")
        break

    recommended_movies_title = recommend_movies(movie_title, 5)

    # Display recommendations based on the provided title
    print("\nRecommended movies based on the provided title:")
    for movie in recommended_movies_title:
        print(f"- {movie}")


# === Recommendation by Preferences ===

while True:
    print("\n=== Recommendation System based on Preferences ===")
    user_input = get_user_preferences_console(df_selected)

    recommended_movies = recommend_by_preferences(
        user_input,
        df_selected,
        vectorizer,
        svd,
        reduced_features,
        num_recommendations = 5
    )

    print("\n🎬 Movies Recommended based on your preferences:")
    for movie in recommended_movies:
        print(f"- {movie}")

    again = input("\nDo you want any further recommendations? (yes/no): ").strip().lower()
    if again.lower() in ['no', 'n']:
        print("Thank you for using the recommendation system!")
        break

    elif again.lower() not in ['yes', 'y']:
       print("Let's start again!")


=== Recommendation System based on Title ===

Enter the title of a movie for similar recommendations (or 'exit' to quit): exit
Thank you for using the recommendation system! Goodbye!

=== Recommendation System based on Preferences ===
Genres available in the dataset:
1. Romance
2. Biography
3. Crime
4. Drama
5. History
6. Adventure
7. Fantasy
8. War
9. Mystery
10. Horror
11. Western
12. Comedy
13. Family
14. Action
15. Sci-Fi
16. Thriller
17. Sport
18. Animation
19. Musical
20. Music
21. Film-Noir
Choose one of the genres above (up to 3 genres), separating by commas: Romance
What is you actor number 1? (If you don't want to choose one, type 'Ignore')Ignore
The user didn't choose any actor.
What is your director number 1? (If you don't want to choose one, type 'Ignore')Ignore
The user didn't choose any director.

🎬 Movies Recommended based on your preferences:
- Stella
- Shiraz
- The Right to Romance
- Cross-Country Romance
- Parineeta

Do you want any further recommendations? (yes/no)

In [ ]:
### RECOMMENDATION SYSTEM WITH EUCLIDEAN DISTANCE

# tokenize text data - to work with text data, tokenization is required
# tokenazation is a way of breaking down the text into smaller units called
# tokens. In this example a token is a word. TDIDF is used to tokenized
# description column. TF-IDF (Term Frequency - Inverse Document Frequency)
# measures how relevant a word is to a document in a collection of documents.abs

# Function to check if a name exists in the dataset (case insensitive)
def name_exists(name, column, df):

    # Convert the name to lowercase for case-insensitive comparison
    all_names = df[column].dropna().apply(lambda x: [n.strip().lower() for n in x.split(', ')]) # removes the missing values
    # of the column; divides the names separated by commas and transforms everything in lowcases, removing spaces

    flat_names = set([n for sublist in all_names for n in sublist]) # creates a list without duplicates

    return name.lower() in flat_names


# Function to suggest full names based on part of the name or surname
def suggest_exact_names(name, column, df):

    # Transform all values in the column into lists of names and surnames, cleaning the spaces
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    # Flatten the list and remove duplicates
    flat_names = list(set([entry for sublist in all_names for entry in sublist]))

    # Use difflib to find similar names
    suggestions = get_close_matches(name, flat_names, n = 5, cutoff = 0.8) # only suggests names with 80% or more similarity

    # If no suggestions are found, search for the corresponding full name
    if not suggestions:
        suggestions = [entry for entry in flat_names if name.lower() in entry.lower()]

    return suggestions

# Function to get the correct name from the dataset, considering capitalization
def get_correct_name(name, column, df):
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    for sublist in all_names: # goes trought all of the names
        for real_name in sublist: # compares the given name with the dataset ones

            if real_name.lower() == name.lower(): # if it matches, returns the name thats on the dataset
                return real_name

    return None # if theres not a match then it returns none

# Function to collect user preferences
def get_user_preferences_console(df):

    # === GENRES ===
    # Show unique genres available in the dataset
    available_genres = df['genre'].str.split(', ').explode().unique()  # Extract unique genres
    print("Genres available in the dataset:")

    for idx, genre in enumerate(available_genres, 1):
        print(f"{idx}. {genre}")

    # Ask the user to choose up to 3 genres
    chosen_genres = []

    while len(chosen_genres) < 1 or len(chosen_genres) > 3:
        genre_choice = input(f"Choose one of the genres above (up to 3 genres), separating by commas: ")

        # Clean and split the input
        genre_list = [genre.strip() for genre in genre_choice.split(",")]

        # Check if all chosen genres are available in the dataset and haven't been chosen yet
        valid_genres = [g for g in genre_list if g in available_genres and g not in chosen_genres]

        # If valid genres
        if valid_genres:
            chosen_genres.extend(valid_genres)

        else:
            print("Invalid choice or genres already selected. Try again.")

        # Limit to 3 genres
        if len(chosen_genres) > 3:
            print("You chose more than 3 genres. We will limit it to 3 genres.")
            chosen_genres = chosen_genres[:3]

    # === ACTORS ===
    chosen_actors = []

    while len(chosen_actors) < 3:
        actor_input = input(f"What is you actor number {len(chosen_actors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if actor_input == "Ignore":
            break

        if name_exists(actor_input, 'actors', df):
            correct_name = get_correct_name(actor_input, 'actors', df)
            chosen_actors.append(correct_name)

        else:
            print(f"The actor '{actor_input}' wasn't found in the dataset.")
            suggestions = suggest_exact_names(actor_input, 'actors', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_actors:
        print("The user didn't choose any actor.")

    else:
        print("Chosen actors:", chosen_actors)

    # === DIRECTOR ===
    chosen_directors = []

    while len(chosen_directors) < 3:
        director_input = input(f"What is your director number {len(chosen_directors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if director_input == "Ignore":
            break

        if name_exists(director_input, 'director', df):
            correct_director = get_correct_name(director_input, 'director', df)
            chosen_directors.append(correct_director)


        else:
            print(f"The director '{director_input}' was not found in the dataset.")
            suggestions = suggest_exact_names(director_input, 'director', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_directors:
        print("The user didn't choose any director.")

    else:
        print("Chosen director:", chosen_directors)


    combined_input = ' '.join(chosen_genres + chosen_actors + chosen_directors)

    return combined_input

"""
    # === LANGUAGES ===
    # Show unique languages available in the dataset
    available_languages = df['language'].str.split(', ').explode().unique()  # Extract unique languages
    print("Languages available in the dataset:")

    for idx, language in enumerate(available_languages, 1):
        print(f"{idx}. {language}")

    # Ask the user to choose up to 3 languages
    chosen_languages = []

    while len(chosen_languages) < 1 or len(chosen_languages) > 3:
        language_choice = input("Choose one of the languages above (up to 3 languages), separating by commas: ")

        # Clean and split the input
        language_list = [lang.strip() for lang in language_choice.split(",")]

        # Check if all chosen languages are available in the dataset and haven't been chosen yet
        valid_languages = [l for l in language_list if l in available_languages and l not in chosen_languages]

        # If valid languages
        if valid_languages:
            chosen_languages.extend(valid_languages)
        else:
            print("Invalid choice or languages already selected. Try again.")

        # Limit to 3 languages
        if len(chosen_languages) > 3:
            print("You chose more than 3 languages. We will limit it to 3 languages.")
            chosen_languages = chosen_languages[:3]


    # === DESCRIPTION ===
    description = input("What type of plot do you prefer? (e.g., Sci-Fi, Romance, Adventure): ").strip()

    # Return all the collected data
    return f"{', '.join(chosen_genres)} {', '.join(chosen_actors)} {', '.join(chosen_directors)} {description}"
"""

# the upper part is not working - IMPROVE THIS SYSTEM

# Prepare data for recommendation
selected_columns = ['original_title', 'genre', 'director', 'actors', 'description']
df_selected = df[selected_columns].astype(str).head(10000)

df_selected['combined_features'] = df_selected.apply(lambda x: ' '.join(x), axis = 1)

# Text Vectorization (TF-IDF)
def create_feature_vectors(df):
    vectorizer = TfidfVectorizer(stop_words = 'english')
    feature_vectors = vectorizer.fit_transform(df['combined_features'])

    # Dimensionality reduction using TruncatedSVD (SVD) to improve performance
    svd = TruncatedSVD(n_components = 300, random_state = 42)
    reduced_features = svd.fit_transform(feature_vectors)

    return vectorizer, svd, feature_vectors, reduced_features

# Function to compute similarity
def compute_similarity_euclidean(reduced_features):
    distances = euclidean_distances(reduced_features)
    similarity = 1 / (1 + distances)  # Convert to similarity-like score
    return similarity


# Vectors and similarity matrix
vectorizer, svd, feature_vectors, reduced_features = create_feature_vectors(df_selected)
similarity_matrix = compute_similarity_euclidean(reduced_features)



# Function to recommend movies based on the title
def recommend_movies(movie_title, num_recommendations = 5):
    if movie_title not in df_selected['original_title'].values:
        # If no movie is found, print suggestions and return an empty list
        print("Movie not found. Try another title." , end = " ")

        # Suggest similar titles if available
        suggestions = get_close_matches(movie_title, df_selected['original_title'].tolist(), n = 5, cutoff = 0.5)

        if suggestions:
            print(", ".join(suggestions))  # Print suggestions in a horizontal format
        else:
            print("No suggestions available.")  # Inform that there are no suggestions

        return []  # Return an empty list instead of None

    movie_index = df_selected[df_selected['original_title'] == movie_title].index[0]
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:num_recommendations + 1]
    recommended_movies = [df_selected.iloc[i[0]]['original_title'] for i in similarity_scores]

    return recommended_movies


# === Main function of recommended by preferences ===

def recommend_by_preferences(user_input, df_selected, vectorizer, svd, feature_vectors, num_recommendations = 5):
    user_vector = vectorizer.transform([user_input])
    user_vector_reduced = svd.transform(user_vector)
    similarity_scores = cosine_similarity(user_vector_reduced, feature_vectors).flatten()
    top_indices = similarity_scores.argsort()[::-1][:num_recommendations]
    recommended_movies = df_selected.iloc[top_indices]['original_title'].tolist()
    return recommended_movies

# === Data preparation ===
# TF-IDF + SVD
vectorizer = TfidfVectorizer(stop_words = 'english')
feature_vectors = vectorizer.fit_transform(df_selected['combined_features'])
svd = TruncatedSVD(n_components = 300, random_state = 42)
reduced_features = svd.fit_transform(feature_vectors)


# === Recommendation by title ===
while True:
    # Ask the user to provide a movie title for recommendations
    print("\n=== Recommendation System based on Title ===")
    movie_title = input("\nEnter the title of a movie for similar recommendations (or 'exit' to quit): ")

    if movie_title.lower() == 'exit':
        print("Thank you for using the recommendation system! Goodbye!")
        break

    recommended_movies_title = recommend_movies(movie_title, 5)

    # Display recommendations based on the provided title
    print("\nRecommended movies based on the provided title:")
    for movie in recommended_movies_title:
        print(f"- {movie}")


# === Recommendation by Preferences ===

while True:
    print("\n=== Recommendation System based on Preferences ===")
    user_input = get_user_preferences_console(df_selected)

    recommended_movies = recommend_by_preferences(
        user_input,
        df_selected,
        vectorizer,
        svd,
        reduced_features,
        num_recommendations = 5
    )

    print("\n🎬 Movies Recommended based on your preferences:")
    for movie in recommended_movies:
        print(f"- {movie}")

    again = input("\nDo you want any further recommendations? (yes/no): ").strip().lower()
    if again.lower() in ['no', 'n']:
        print("Thank you for using the recommendation system!")
        break

    elif again.lower() not in ['yes', 'y']:
       print("Let's start again!")

In [ ]:
### RECOMMENDATION SYSTEM WITH PEARSON CORRELATION

# tokenize text data - to work with text data, tokenization is required
# tokenazation is a way of breaking down the text into smaller units called
# tokens. In this example a token is a word. TDIDF is used to tokenized
# description column. TF-IDF (Term Frequency - Inverse Document Frequency)
# measures how relevant a word is to a document in a collection of documents.abs

# Function to check if a name exists in the dataset (case insensitive)
def name_exists(name, column, df):

    # Convert the name to lowercase for case-insensitive comparison
    all_names = df[column].dropna().apply(lambda x: [n.strip().lower() for n in x.split(', ')]) # removes the missing values
    # of the column; divides the names separated by commas and transforms everything in lowcases, removing spaces

    flat_names = set([n for sublist in all_names for n in sublist]) # creates a list without duplicates

    return name.lower() in flat_names


# Function to suggest full names based on part of the name or surname
def suggest_exact_names(name, column, df):

    # Transform all values in the column into lists of names and surnames, cleaning the spaces
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    # Flatten the list and remove duplicates
    flat_names = list(set([entry for sublist in all_names for entry in sublist]))

    # Use difflib to find similar names
    suggestions = get_close_matches(name, flat_names, n = 5, cutoff = 0.8) # only suggests names with 80% or more similarity

    # If no suggestions are found, search for the corresponding full name
    if not suggestions:
        suggestions = [entry for entry in flat_names if name.lower() in entry.lower()]

    return suggestions

# Function to get the correct name from the dataset, considering capitalization
def get_correct_name(name, column, df):
    all_names = df[column].dropna().apply(lambda x: [n.strip() for n in x.split(', ')])

    for sublist in all_names: # goes trought all of the names
        for real_name in sublist: # compares the given name with the dataset ones

            if real_name.lower() == name.lower(): # if it matches, returns the name thats on the dataset
                return real_name

    return None # if theres not a match then it returns none

# Function to collect user preferences
def get_user_preferences_console(df):

    # === GENRES ===
    # Show unique genres available in the dataset
    available_genres = df['genre'].str.split(', ').explode().unique()  # Extract unique genres
    print("Genres available in the dataset:")

    for idx, genre in enumerate(available_genres, 1):
        print(f"{idx}. {genre}")

    # Ask the user to choose up to 3 genres
    chosen_genres = []

    while len(chosen_genres) < 1 or len(chosen_genres) > 3:
        genre_choice = input(f"Choose one of the genres above (up to 3 genres), separating by commas: ")

        # Clean and split the input
        genre_list = [genre.strip() for genre in genre_choice.split(",")]

        # Check if all chosen genres are available in the dataset and haven't been chosen yet
        valid_genres = [g for g in genre_list if g in available_genres and g not in chosen_genres]

        # If valid genres
        if valid_genres:
            chosen_genres.extend(valid_genres)

        else:
            print("Invalid choice or genres already selected. Try again.")

        # Limit to 3 genres
        if len(chosen_genres) > 3:
            print("You chose more than 3 genres. We will limit it to 3 genres.")
            chosen_genres = chosen_genres[:3]

    # === ACTORS ===
    chosen_actors = []

    while len(chosen_actors) < 3:
        actor_input = input(f"What is you actor number {len(chosen_actors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if actor_input == "Ignore":
            break

        if name_exists(actor_input, 'actors', df):
            correct_name = get_correct_name(actor_input, 'actors', df)
            chosen_actors.append(correct_name)

        else:
            print(f"The actor '{actor_input}' wasn't found in the dataset.")
            suggestions = suggest_exact_names(actor_input, 'actors', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_actors:
        print("The user didn't choose any actor.")

    else:
        print("Chosen actors:", chosen_actors)

    # === DIRECTOR ===
    chosen_directors = []

    while len(chosen_directors) < 3:
        director_input = input(f"What is your director number {len(chosen_directors) + 1}? (If you don't want to choose one, type 'Ignore')").strip()

        if director_input == "Ignore":
            break

        if name_exists(director_input, 'director', df):
            correct_director = get_correct_name(director_input, 'director', df)
            chosen_directors.append(correct_director)


        else:
            print(f"The director '{director_input}' was not found in the dataset.")
            suggestions = suggest_exact_names(director_input, 'director', df)

            if suggestions:
                print("Suggestions:", suggestions)

            else:
                print("No suggestions found. Try another name.")

    if not chosen_directors:
        print("The user didn't choose any director.")

    else:
        print("Chosen director:", chosen_directors)


    combined_input = ' '.join(chosen_genres + chosen_actors + chosen_directors)

    return combined_input

"""
    # === LANGUAGES ===
    # Show unique languages available in the dataset
    available_languages = df['language'].str.split(', ').explode().unique()  # Extract unique languages
    print("Languages available in the dataset:")

    for idx, language in enumerate(available_languages, 1):
        print(f"{idx}. {language}")

    # Ask the user to choose up to 3 languages
    chosen_languages = []

    while len(chosen_languages) < 1 or len(chosen_languages) > 3:
        language_choice = input("Choose one of the languages above (up to 3 languages), separating by commas: ")

        # Clean and split the input
        language_list = [lang.strip() for lang in language_choice.split(",")]

        # Check if all chosen languages are available in the dataset and haven't been chosen yet
        valid_languages = [l for l in language_list if l in available_languages and l not in chosen_languages]

        # If valid languages
        if valid_languages:
            chosen_languages.extend(valid_languages)
        else:
            print("Invalid choice or languages already selected. Try again.")

        # Limit to 3 languages
        if len(chosen_languages) > 3:
            print("You chose more than 3 languages. We will limit it to 3 languages.")
            chosen_languages = chosen_languages[:3]


    # === DESCRIPTION ===
    description = input("What type of plot do you prefer? (e.g., Sci-Fi, Romance, Adventure): ").strip()

    # Return all the collected data
    return f"{', '.join(chosen_genres)} {', '.join(chosen_actors)} {', '.join(chosen_directors)} {description}"
"""

# the upper part is not working - IMPROVE THIS SYSTEM

# Prepare data for recommendation
selected_columns = ['original_title', 'genre', 'director', 'actors', 'description']
df_selected = df[selected_columns].astype(str).head(10000)

df_selected['combined_features'] = df_selected.apply(lambda x: ' '.join(x), axis = 1)

# Text Vectorization (TF-IDF)
def create_feature_vectors(df):
    vectorizer = TfidfVectorizer(stop_words = 'english')
    feature_vectors = vectorizer.fit_transform(df['combined_features'])

    # Dimensionality reduction using TruncatedSVD (SVD) to improve performance
    svd = TruncatedSVD(n_components = 300, random_state = 42)
    reduced_features = svd.fit_transform(feature_vectors)

    return vectorizer, svd, feature_vectors, reduced_features

# Function to compute similarity
def compute_similarity_euclidean(reduced_features):
    distances = euclidean_distances(reduced_features)
    similarity = 1 / (1 + distances)  # Convert to similarity-like score
    return similarity


# Vectors and similarity matrix
vectorizer, svd, feature_vectors, reduced_features = create_feature_vectors(df_selected)
similarity_matrix = compute_similarity_euclidean(reduced_features)



# Function to recommend movies based on the title
def recommend_movies(movie_title, num_recommendations = 5):
    if movie_title not in df_selected['original_title'].values:
        # If no movie is found, print suggestions and return an empty list
        print("Movie not found. Try another title." , end = " ")

        # Suggest similar titles if available
        suggestions = get_close_matches(movie_title, df_selected['original_title'].tolist(), n = 5, cutoff = 0.5)

        if suggestions:
            print(", ".join(suggestions))  # Print suggestions in a horizontal format
        else:
            print("No suggestions available.")  # Inform that there are no suggestions

        return []  # Return an empty list instead of None

    movie_index = df_selected[df_selected['original_title'] == movie_title].index[0]
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:num_recommendations + 1]
    recommended_movies = [df_selected.iloc[i[0]]['original_title'] for i in similarity_scores]

    return recommended_movies


# === Main function of recommended by preferences ===

def recommend_by_preferences(user_input, df_selected, vectorizer, svd, feature_vectors, num_recommendations = 5):
    user_vector = vectorizer.transform([user_input])
    user_vector_reduced = svd.transform(user_vector)
    similarity_scores = cosine_similarity(user_vector_reduced, feature_vectors).flatten()
    top_indices = similarity_scores.argsort()[::-1][:num_recommendations]
    recommended_movies = df_selected.iloc[top_indices]['original_title'].tolist()
    return recommended_movies

# === Data preparation ===
# TF-IDF + SVD
vectorizer = TfidfVectorizer(stop_words = 'english')
feature_vectors = vectorizer.fit_transform(df_selected['combined_features'])
svd = TruncatedSVD(n_components = 300, random_state = 42)
reduced_features = svd.fit_transform(feature_vectors)


# === Recommendation by title ===
while True:
    # Ask the user to provide a movie title for recommendations
    print("\n=== Recommendation System based on Title ===")
    movie_title = input("\nEnter the title of a movie for similar recommendations (or 'exit' to quit): ")

    if movie_title.lower() == 'exit':
        print("Thank you for using the recommendation system! Goodbye!")
        break

    recommended_movies_title = recommend_movies(movie_title, 5)

    # Display recommendations based on the provided title
    print("\nRecommended movies based on the provided title:")
    for movie in recommended_movies_title:
        print(f"- {movie}")


# === Recommendation by Preferences ===

while True:
    print("\n=== Recommendation System based on Preferences ===")
    user_input = get_user_preferences_console(df_selected)

    recommended_movies = recommend_by_preferences(
        user_input,
        df_selected,
        vectorizer,
        svd,
        reduced_features,
        num_recommendations = 5
    )

    print("\n🎬 Movies Recommended based on your preferences:")
    for movie in recommended_movies:
        print(f"- {movie}")

    again = input("\nDo you want any further recommendations? (yes/no): ").strip().lower()
    if again.lower() in ['no', 'n']:
        print("Thank you for using the recommendation system!")
        break

    elif again.lower() not in ['yes', 'y']:
       print("Let's start again!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.stats import pearsonr
from sklearn.metrics import precision_score, recall_score, f1_score
from difflib import get_close_matches
import warnings
warnings.filterwarnings("ignore")

# ========== DATA LOADING AND PREPARATION ========== #

# Load your DataFrame (replace with your path if needed)
# Example: df = pd.read_csv("your_movies_dataset.csv")
# Use a sample of the data
selected_columns = ['original_title', 'genre', 'director', 'actors', 'description']
df_selected = df[selected_columns].astype(str).head(10000)
df_selected['combined_features'] = df_selected.apply(lambda x: ' '.join(x), axis=1)

# ========== FEATURE VECTORIZATION ========== #
def create_feature_vectors(df):
    vectorizer = TfidfVectorizer(stop_words='english')
    feature_vectors = vectorizer.fit_transform(df['combined_features'])

    svd = TruncatedSVD(n_components=300, random_state=42)
    reduced_features = svd.fit_transform(feature_vectors)

    return vectorizer, svd, feature_vectors, reduced_features

vectorizer, svd, feature_vectors, reduced_features = create_feature_vectors(df_selected)

# ========== SIMILARITY CALCULATION ========== #
def calculate_similarity(user_vector_reduced, feature_vectors_reduced, method='cosine'):
    if method == 'cosine':
        similarity_scores = cosine_similarity(user_vector_reduced, feature_vectors_reduced).flatten()
    elif method == 'euclidean':
        distances = euclidean_distances(user_vector_reduced, feature_vectors_reduced).flatten()
        similarity_scores = 1 / (1 + distances)
    elif method == 'pearson':
        similarity_scores = []
        user_vector = user_vector_reduced.flatten()
        for vec in feature_vectors_reduced:
            corr, _ = pearsonr(user_vector, vec)
            similarity_scores.append(corr if not np.isnan(corr) else 0)
        similarity_scores = np.array(similarity_scores)
    else:
        raise ValueError("Invalid similarity method")
    return similarity_scores

# ========== RECOMMENDATION FUNCTION ========== #
def recommend_by_preferences(user_input, df_selected, vectorizer, svd, reduced_features, num_recommendations = 5, method = 'cosine'):
    user_vector = vectorizer.transform([user_input])
    user_vector_reduced = svd.transform(user_vector)
    similarity_scores = calculate_similarity(user_vector_reduced, reduced_features, method)
    top_indices = similarity_scores.argsort()[::-1][:num_recommendations]
    recommended_movies = df_selected.iloc[top_indices]['original_title'].tolist()
    return recommended_movies

# ========== EVALUATION FUNCTION ========== #
def evaluate_similarity_method(method, df_selected, vectorizer, svd, reduced_features, num_tests = 100, k = 5):
    y_true_all = []
    y_pred_all = []

    test_movies = df_selected['original_title'].sample(num_tests, random_state = 42).tolist()

    for title in test_movies:
        row = df_selected[df_selected['original_title'] == title].iloc[0]
        input_str = ' '.join([row['genre'], row['director'], row['actors']])  # simulated preference

        recommendations = recommend_by_preferences(
            input_str,
            df_selected,
            vectorizer,
            svd,
            reduced_features,
            num_recommendations = k,
            method=method
        )

        y_true = [1]
        y_pred = [1 if title in recommendations else 0]

        y_true_all.extend(y_true)
        y_pred_all.extend(y_pred)

    precision = precision_score(y_true_all, y_pred_all, zero_division = 0)
    recall = recall_score(y_true_all, y_pred_all, zero_division = 0)
    f1 = f1_score(y_true_all, y_pred_all, zero_division = 0)

    return precision, recall, f1

# ========== RUN EVALUATION ========== #
def run_all_evaluations(df_selected, vectorizer, svd, reduced_features):
    methods = ['cosine', 'euclidean', 'pearson']
    print("=== Similarity Evaluation (Precision, Recall, F1) ===\n")
    for method in methods:
        precision, recall, f1 = evaluate_similarity_method(
            method,
            df_selected,
            vectorizer,
            svd,
            reduced_features,
            num_tests = 100,
            k = 5
        )
        print(f"{method.capitalize()} Similarity:")
        print(f" - Precision: {precision:.2f}")
        print(f" - Recall:    {recall:.2f}")
        print(f" - F1 Score:  {f1:.2f}\n")

# ========== EXECUTE ========== #
run_all_evaluations(df_selected, vectorizer, svd, reduced_features)


=== Similarity Evaluation (Precision, Recall, F1) ===

Cosine Similarity:
 - Precision: 1.00
 - Recall:    0.99
 - F1 Score:  0.99

Euclidean Similarity:
 - Precision: 1.00
 - Recall:    0.98
 - F1 Score:  0.99

Pearson Similarity:
 - Precision: 1.00
 - Recall:    0.99
 - F1 Score:  0.99



In [ ]:
# EVALUATION METRICS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# Data Prep
df = df_selected[['description', 'genre']].dropna()

# Convert genre strings into lists (split by comma)
df['genre'] = df['genre'].apply(lambda x: [genre.strip() for genre in x.split(',')])

# Binarize the multi-label genres
mlb = MultiLabelBinarizer()            # Creates an instance of the encoder; This code uses MultiLabelBinarizer to convert a
                                       # column containing multiple labels per entry into a binary (one-hot encoded) format
y = mlb.fit_transform(df['genre'])     # Transforms the 'genre' column into binary vectors


# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], y, test_size = 0.2, random_state = 42
)

# Create pipeline: TF-IDF + Multi-output Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', max_features = 5000)),
    ('clf', MultiOutputClassifier(LogisticRegression(max_iter = 1000)))
])

# Train the model
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names = mlb.classes_))

# Metrics Description:

# Precision: Out of all the times the model predicted this class, how many were correct? Formula: TP / (TP + FP)
#            example: Precision = 0.43 → When the model predicted this class, it was right 43% of the time.

# Recall: Out of all the actual examples of this class, how many did the model correctly find? Formula: TP / (TP + FN)
#         example: Recall = 0.02 → Out of 191 real examples of this class, it only found 2%.

# F1-score: The harmonic mean of precision and recall. A balanced measure when you care about both false positives
#           and false negatives. Formula: 2 * (precision * recall) / (precision + recall)
#           example: F1-score = 0.03 → Low overall performance for this class (poor balance between precision and recall).

# Support: The number of real examples of this class in the test set.
#          example: Support = 191 → There were 191 actual examples of this class in the test set.

Classification Report:
              precision    recall  f1-score   support

      Action       0.43      0.02      0.03       191
   Adventure       0.79      0.10      0.18       291
   Animation       0.00      0.00      0.00        10
   Biography       0.75      0.05      0.10        56
      Comedy       0.73      0.36      0.49       608
       Crime       0.78      0.27      0.40       360
       Drama       0.69      0.79      0.74      1131
      Family       0.00      0.00      0.00        61
     Fantasy       0.00      0.00      0.00        47
   Film-Noir       0.25      0.01      0.01       135
     History       0.00      0.00      0.00        83
      Horror       1.00      0.06      0.12        95
       Music       0.00      0.00      0.00        80
     Musical       0.00      0.00      0.00       140
     Mystery       0.86      0.08      0.14       152
     Romance       0.60      0.15      0.25       531
      Sci-Fi       1.00      0.04      0.07        55
    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# GRID SEARCH
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
from scipy.stats import randint

# Data Prep
df = df_selected[['description', 'genre']].dropna()

# Convert genre strings into lists (split by comma)
df['genre'] = df['genre'].apply(lambda x: [genre.strip() for genre in x.split(',')])

# Binarize the multi-label genres
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genre'])

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], y, test_size = 0.2, random_state = 42
)

# Define pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', max_features = 5000)),
    ('clf', MultiOutputClassifier(LogisticRegression(max_iter = 4000, class_weight = 'balanced')))
    # added class_weight = 'balanced' because the Animation genre was being ignored, but decreased my
    # precision and improved my recall; The model has 1.00 precision for Animation because it only predicted
    # it once and got it right, but missed 9 out of 10 actual cases, leading to low recall; f1-score is low
    # because of the prediction and the recall; the number of iterations increased because of a warning its
    # giving me, but the warning is still in the output (see this)
])

# Parameter grid for GridSearchCV
param_grid = {
    'tfidf__max_features': [15200, 15500, 15700], # limits the number of words TfidfVectorizer will consider;
                                                  # Too few = not enough information; too many = more noise, slower training;
                                                  # we have 85000 rows so we are trying on these values

    'clf__estimator__C': [155, 160, 165],            # Controls the regularization strength of LogisticRegression;
                                                     # Lower C = stronger regularization (simpler model);
                                                     # higher C = weaker regularization (more flexible model)

    'clf__estimator__max_iter': [70, 75, 80]   # Sets the maximum number of iterations for the solver to find the best model;
                                               # Logistic Regression uses optimization — if your data is large or complex,
                                               # it may need more steps to converge.
}

# When I had this parameters:

# 'tfidf__max_features': [5000, 10000, 20000] -> ... -> [9000, 10000, 11000] -> [10000, 12000, 15000]
#                        -> [14000, 16000, 17000] -> [15500, 16000, 16500]
# 'clf__estimator__C': [500, 1000, 2000] -> [50, 100, 200] -> [100, 150, 200] -> [50, 100, 150] -> [130, 150, 170]
#                      -> [145, 150, 155]
# 'clf__estimator__max_iter': [500, 1000, 2000] -> ... -> [300, 350, 400] -> [100, 200, 300] -> [50, 100, 150]
#                             -> [80, 100, 110]

# I got this:
# Best parameters found: {'clf__estimator__C': 100, 'clf__estimator__max_iter': 500, 'tfidf__max_features': 10000}
# Best parameters found: {'clf__estimator__C': 200, 'clf__estimator__max_iter': 500, 'tfidf__max_features': 8000}
# Best parameters found: {'clf__estimator__C': 100, 'clf__estimator__max_iter': 300, 'tfidf__max_features': 11000}
# Best parameters found: {'clf__estimator__C': 150, 'clf__estimator__max_iter': 100, 'tfidf__max_features': 15000}
# Best parameters found: {'clf__estimator__C': 150, 'clf__estimator__max_iter': 100, 'tfidf__max_features': 16000}
# Best parameters found: {'clf__estimator__C': 155, 'clf__estimator__max_iter': 80, 'tfidf__max_features': 15500}


# GridSearchCV setup
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv = 3,
    scoring = 'f1_micro', # changed the scoring from f1_micro (o melhor) -> f1_weighted = f1_macro
                              # -> precision_micro (melhora todos (a precison fica muito boa com este) e piora o recall)
                              # -> recall_micro (melhora o recall mas piora o f1-score e mantém o precision)
    verbose = 2,
    n_jobs = -1
)

# Fit the model with grid search
grid_search.fit(X_train, y_train)

# Predict on test set with best estimator
y_pred = grid_search.best_estimator_.predict(X_test)

# Evaluation
print("Best parameters found:", grid_search.best_params_)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names = mlb.classes_))

# micro_avg: Calculates metrics globally — it sums up all true positives, false positives,
#            and false negatives across all labels, then computes precision/recall/F1.
#            example: F1 = 0.50 means the model gets half of the label predictions right overall.

# macro_avg: Calculates precision, recall, and F1 individually for each label, then takes
#            the unweighted average (treats all genres equally).
#            example: F1 = 0.33 is lower than micro because the model may be bad at rare genres

# weighted_avg: Like macro, but weights each label by how often it appears;
#               example: F1 = 0.48 — closer to micro F1, because frequent genres have more weight

# samples_avg: For each individual sample, it calculates:

#              precision = how many predicted genres were correct,
#              recall = how many of the true genres were found,
#              F1-score = harmonic mean of those two.
#              Then it averages these F1-scores across all samples.

Fitting 3 folds for each of 27 candidates, totalling 81 fits


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Best parameters found: {'clf__estimator__C': 155, 'clf__estimator__max_iter': 70, 'tfidf__max_features': 15200}
Classification Report:
              precision    recall  f1-score   support

      Action       0.31      0.27      0.29       191
   Adventure       0.46      0.38      0.42       291
   Animation       1.00      0.10      0.18        10
   Biography       0.29      0.29      0.29        56
      Comedy       0.59      0.57      0.58       608
       Crime       0.59      0.53      0.56       360
       Drama       0.69      0.70      0.69      1131
      Family       0.35      0.10      0.15        61
     Fantasy       0.27      0.30      0.28        47
   Film-Noir       0.33      0.29      0.31       135
     History       0.48      0.25      0.33        83
      Horror       0.55      0.39      0.46        95
       Music       0.16      0.10      0.12        80
     Musical       0.33      0.31      0.32       140
     Mystery       0.46      0.38      0.41       152


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# RANDOM SEARCH
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
from scipy.stats import randint

# Data preparation
df = df_selected[['description', 'genre']].dropna()

# Convert genre strings into lists (split by comma)
df['genre'] = df['genre'].apply(lambda x: [genre.strip() for genre in x.split(',')])

# Binarize the multi-label genres
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genre'])

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], y, test_size = 0.2, random_state = 42
)

# Modeling pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', max_features = 5000)),
    ('clf', MultiOutputClassifier(
        LogisticRegression(max_iter = 4000, class_weight = 'balanced')))
])

# Parameter distribution for RandomizedSearchCV
param_dist = {
    'tfidf__max_features': randint(14000, 17000),
    'clf__estimator__C': randint(100, 200),
    'clf__estimator__max_iter': randint(50, 150)
}

# RandomizedSearchCV configuration
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions = param_dist,
    n_iter = 20,             # number of random combinations to try
    cv = 3,
    scoring = 'f1_micro',
    verbose = 2,
    random_state = 42,
    n_jobs = -1
)

# Train model
random_search.fit(X_train, y_train)

# Evaluation
y_pred = random_search.best_estimator_.predict(X_test)

print("Best parameters found:", random_search.best_params_)
print("Classification report:")
print(classification_report(y_test, y_pred, target_names = mlb.classes_))

# usar apenas as avg - no relatório

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters found: {'clf__estimator__C': 151, 'clf__estimator__max_iter': 142, 'tfidf__max_features': 15294}
Classification report:
              precision    recall  f1-score   support

      Action       0.32      0.27      0.29       191
   Adventure       0.45      0.38      0.41       291
   Animation       1.00      0.10      0.18        10
   Biography       0.29      0.29      0.29        56
      Comedy       0.59      0.57      0.58       608
       Crime       0.59      0.54      0.56       360
       Drama       0.68      0.70      0.69      1131
      Family       0.35      0.10      0.15        61
     Fantasy       0.28      0.30      0.29        47
   Film-Noir       0.38      0.30      0.34       135
     History       0.49      0.28      0.35        83
      Horror       0.55      0.39      0.46        95
       Music       0.14      0.09      0.11        80
     Musical       0.33      0.31      0.32   

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
